<a href="https://colab.research.google.com/github/juanitarhea/ai-drone-swarm/blob/main/M2_SurvivorNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics -q

from google.colab import files
uploaded = files.upload()

import zipfile, os

zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/dataset")

# Fix val → valid if needed
if os.path.exists("/content/dataset/val") and not os.path.exists("/content/dataset/valid"):
    os.rename("/content/dataset/val", "/content/dataset/valid")

# Check structure
for root, dirs, files in os.walk("/content/dataset"):
    print(root)

Saving Person Detection.yolov8 (1).zip to Person Detection.yolov8 (1) (2).zip
/content/dataset
/content/dataset/valid
/content/dataset/valid/images
/content/dataset/valid/labels
/content/dataset/train
/content/dataset/train/images
/content/dataset/train/labels


In [ ]:
import os
import shutil
import random

train_img = "/content/dataset/train/images"
train_lbl = "/content/dataset/train/labels"

val_img = "/content/dataset/valid/images"
val_lbl = "/content/dataset/valid/labels"

os.makedirs(val_img, exist_ok=True)
os.makedirs(val_lbl, exist_ok=True)

images = os.listdir(train_img)
random.shuffle(images)

split = int(0.1 * len(images))

for img in images[:split]:
    img_path = os.path.join(train_img, img)
    new_img_path = os.path.join(val_img, img)

    if os.path.exists(img_path):
        shutil.move(img_path, new_img_path)

    label = img.rsplit(".", 1)[0] + ".txt"
    lbl_path = os.path.join(train_lbl, label)
    new_lbl_path = os.path.join(val_lbl, label)

    if os.path.exists(lbl_path):
        shutil.move(lbl_path, new_lbl_path)

print("Dataset split complete ✅")

Dataset split complete ✅


In [ ]:
from ultralytics import YOLO
import yaml

model = YOLO("yolov8n.pt")
print("Model loaded ✅")

# Define the content for dataset.yaml
dataset_yaml_content = {
    'path': '/content/dataset', # Dataset root directory
    'train': 'train/images', # Train images relative to 'path'
    'val': 'valid/images',   # Val images relative to 'path'
    'nc': 1,                 # Number of classes
    'names': ['person']      # Class names
}

# Specify the path for the dataset.yaml file
dataset_yaml_path = '/content/dataset.yaml'

# Create and write the dataset.yaml file
with open(dataset_yaml_path, 'w') as f:
    yaml.dump(dataset_yaml_content, f, default_flow_style=False)

print(f"Dataset configuration saved to {dataset_yaml_path} ✅")

model.train(
    data=dataset_yaml_path,
    epochs=20,
    imgsz=416,
    batch=8,
    project="SurvivorNet"
)

metrics = model.val()
print(metrics)


Model loaded ✅
Dataset configuration saved to /content/dataset.yaml ✅
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train7, nbs=64, nms=False, opset=None, op

In [ ]:
from IPython.display import Image

Image("/content/runs/detect/SurvivorNet/train/results.png")